# H-Ngram — PoC одной L2-субтаблицы поверх AtomicNG

**Иерархическая внешняя память**: «процессор» (замороженное ядро) + «библиотека знаний»
(L2-субтаблица на диске, подгружаемая в GPU-память по теме диалога).

Что показывает ноутбук (весь маршрут концепции H-Ngram на одном томе):

1. адресация n-грамм на устройстве: blake2b (совместимость с ядром) ↔ poly64 (GPU-кернел);
2. наполнение субтаблицы через D1 (та же семантика записи, что у патча AtomicNG);
3. диск: int8-квантование + integrity-манифест; LRU-менеджер томов;
4. zero-init gate: attach памяти не меняет выходы ядра бит-в-бит;
5. PoC: адаптер учится потреблять Δ-строки → NLL на домене падает, нейтраль не тронута;
6. роутер темы с гистерезисом (без ping-pong-переключений);
7. L0-ярус KVOff: KV-кэш, выгружаемый в таблицу для освобождения памяти — точное
   восстановление бит-в-бит, recompute-fallback, дистилляция gist в D1-строки.

Ядро — `TorchTinyModel` (наполнитель/адаптер) и `KVTinyModel` (настоящий причинный
attention с явным KV-кэшем для KVOff), CPU/CUDA, без внешних зависимостей.
На GPU меняется одна строка: `DEVICE`.


In [ ]:
# В репо: pip install -e ".[torch]" · запуск из notebooks/ — путь в src без установки
import os, sys, time

sys.path.insert(0, os.path.join(os.getcwd(), os.pardir, "src"))

import atomicng
import torch
from atomicng.hngram.fill import fill_subtable_from_sequences
from atomicng.hngram.hashing import addresses, blake_addresses, poly_addresses
from atomicng.hngram.manager import SubtableManager
from atomicng.hngram.router import TopicRouter
from atomicng.hngram.sidecar import L2Sidecar
from atomicng.hngram.store import L2Subtable, SubtableSpec
from atomicng.writers.training import TorchTinyModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("atomicng", atomicng.__version__, "| torch", torch.__version__, "| device:", DEVICE)

## 1. Адресация n-грамм на устройстве

Две схемы хеша: **blake2b** — канонический путь NumPy-ядра (TinyModel.row_hash), считается
на CPU; **poly64** — векторизованный кернел (катящийся полином + splitmix-финализатор на
int64), живёт на GPU. Субтаблица фиксирует схему в спецификации: запись и чтение обязаны
совпадать (инвариант).


In [ ]:
VOCAB, D, N_ROWS = 64, 32, 256

ids = torch.randint(0, VOCAB, (4, 20), dtype=torch.long)
rows_poly = poly_addresses(ids, N_ROWS, key_order=2)
print("poly64:", tuple(rows_poly.shape), "детерминизм:", torch.equal(rows_poly, poly_addresses(ids, N_ROWS, 2)))

# blake2b-путь совпадает с ядром
seq = [3, 17, 5, 200, 64]
ok = all(blake_addresses(seq, N_ROWS, 2)[w] == atomicng.TinyModel.row_hash(seq[w], seq[w + 1], N_ROWS)
         for w in range(len(seq) - 1))
print("blake2b == TinyModel.row_hash:", ok)

# покрытие: случайные окна заполняют таблицу широко (нет вырождения кернела)
big = torch.randint(0, VOCAB, (64, 64), dtype=torch.long)
distinct = len(set(poly_addresses(big, 1024, 2).flatten().tolist()))
print(f"покрытие 4096 окон → {distinct}/1024 строк")

## 2. Наполнение L2-субтаблицы через D1

Строка биграммы (tᵢ₋₁, tᵢ) хранит **продолжение** tᵢ₊₁ (delta-смысл строки: «что обычно
следует дальше»). События записываются правилом D1 из ядра AtomicNG (блок-усреднение,
gate ∈ [0,1]) — субтаблица это материалезованный MemoryPatch в плотной форме.


In [ ]:
SPEC = SubtableSpec(name="domain-l2", n_rows=N_ROWS, row_width=D, key_order=2, hash_scheme="blake2b")

FACT = [5, 9, 13, 21, 33, 41]                       # «факт» домена (стиль test_torch)
NOISE = [2, 7, 11, 19, 23, 29, 31, 37, 43, 47]

def corpus(n=8, seed=7):
    gen = torch.Generator().manual_seed(seed)
    seqs = []
    for _ in range(n):
        head = [int(NOISE[int(torch.randint(0, len(NOISE), (1,), generator=gen))]) for _ in range(6)]
        tail = [int(NOISE[int(torch.randint(0, len(NOISE), (1,), generator=gen))]) for _ in range(6)]
        seqs.append(head + FACT + tail)
    return seqs

sub = fill_subtable_from_sequences(corpus(), SPEC, VOCAB)   # D1-наполнение → плотный том
st = sub.stats()
print({k: round(v, 4) if isinstance(v, float) else v for k, v in st.items()})

# покрытие: окна корпуса записи бьют в том со 100%, случайные — на уровне плотности
same_corpus = [r for seq in corpus(4) for r in blake_addresses(seq, N_ROWS, 2)[:-1]]
rand_rows = torch.randint(0, N_ROWS, (256,), generator=torch.Generator().manual_seed(5))
print("hit_rate корпуса:", sub.hit_rate(torch.tensor(same_corpus)),
      "| случайный:", round(sub.hit_rate(rand_rows), 3), "| плотность:", round(st["fill_ratio"], 3))

## 3. Диск (int8 + integrity) и LRU-менеджер томов

Том квантуется в int8 с глобальным масштабом и сохраняется с sha256-манифестом — конвенции
overlay AtomicNG. Менеджер держит на устройстве не более `n_slots` томов: при смене темы
лишний вытесняется (LRU), новый подгружается с «диска»; время подгрузки — в статистике.


In [ ]:
import tempfile

tmp = tempfile.mkdtemp(prefix="hngram-poc-")
sub.save_pretrained(os.path.join(tmp, "domain-l2.npz"))
loaded = L2Subtable.load_pretrained(os.path.join(tmp, "domain-l2.npz"), device=DEVICE)
tol = max(float(sub.values.abs().max()) / 127.0, 1e-6)
print("int8-roundtrip: max err =", float((loaded.values - sub.values).abs().max()), "≤ tol =", round(tol, 6))

# tamper → integrity-манифест спотыкается (строгая загрузка, как у overlay)
import numpy as np

path = os.path.join(tmp, "domain-l2.npz")
with np.load(path, allow_pickle=False) as data:
    payload = {k: data[k] for k in data.files}
payload["values_i8"][0, 0] += 7
np.savez(os.path.join(tmp, "tampered.npz"), **payload)
try:
    L2Subtable.load_pretrained(os.path.join(tmp, "tampered.npz"))
    print("tamper НЕ пойман — баг!")
except atomicng.OverlayError as e:
    print("tamper пойман:", e)

# LRU: три тома на двух слотах → третий вытесняет первый
manager = SubtableManager(os.path.join(tmp, "store"), n_slots=2, device=DEVICE)
for i, name in enumerate(["code", "med", "law"]):
    spec = SubtableSpec(name=name, n_rows=N_ROWS, row_width=D, key_order=2, hash_scheme="blake2b")
    seqs = [[(t + 8 * i) % VOCAB for t in seq] for seq in corpus(2, seed=7 + i)]
    manager.register(fill_subtable_from_sequences(seqs, spec, VOCAB))

_, ev1, ms1 = manager.ensure("code")
_, ev2, ms2 = manager.ensure("med")
_, ev3, ms3 = manager.ensure("law")
print("3 тома на 2 слотах: подгрузка law вытеснила", ev3, f"| load {ms3:.1f} ms")
print(manager.stats_summary())

## 4. Zero-init gate: безопасный attach

Sidecar добавляет Δ_L2 = γ ⊙ Proj(Δ_row) в резидуальный поток **замороженного** ядра.
При attach γ = 0 → выходы бит-в-бит равны базовым: подключение памяти не может
сломать модель до обучения (аналог kill-switch'а AtomicNG).


In [ ]:
model = TorchTinyModel(vocab_size=VOCAB, hidden_size=D, n_rows=N_ROWS, seed=17).to(DEVICE)
manager = SubtableManager(os.path.join(tmp, "store"), n_slots=1, device=DEVICE)
manager.register(sub)
manager.pin("domain-l2")            # ручная фиксация тома (роутер — в §6)
sidecar = L2Sidecar.attach(model, manager)

ids = torch.randint(0, VOCAB, (2, 14), dtype=torch.long)
base_h, base_logits = model(ids, torch.zeros(2, 13, D))
h, logits = sidecar(ids)
print("γ при attach:", sidecar.gate_norm(), "| hidden бит-в-бит:", torch.equal(h, base_h),
      "| logits бит-в-бит:", torch.equal(logits, base_logits))

## 5. PoC: адаптер учится потреблять Δ-строки

Обучаются **только** γ и Proj (ядро и том заморожены). Ожидание: NLL на доменных пробах
падает; на нейтральных пробах (чьи биграммы не бьют в том) — не меняется ни на бит:
негативный контроль, строки вне тома дают нулевую Δ.


In [ ]:
def nll(sc, seqs):
    total, count = 0.0, 0
    for seq in seqs:
        ids_ = torch.tensor([seq], dtype=torch.long, device=DEVICE)
        _, lg = sc(ids_)
        total += float(torch.nn.functional.cross_entropy(lg[0], ids_[0, 1:], reduction="sum").detach())
        count += len(seq) - 1
    return total / max(count, 1)

facts = corpus()
occupied = set(torch.nonzero(sub.occupancy).flatten().tolist())
gen = torch.Generator().manual_seed(99)
neutral, tries = [], 0
while len(neutral) < 6 and tries < 5000:
    tries += 1
    seq = torch.randint(0, VOCAB, (14,), generator=gen).tolist()
    if occupied.isdisjoint(blake_addresses(seq, N_ROWS, 2)):
        neutral.append([int(t) for t in seq])

t0 = time.perf_counter()
nll_domain_0, nll_neutral_0 = nll(sidecar, facts), nll(sidecar, neutral)

opt = torch.optim.AdamW(sidecar.parameters(), lr=0.1)
batch = torch.tensor(facts, dtype=torch.long, device=DEVICE)
for _ in range(80):
    _, lg = sidecar(batch)
    loss = torch.nn.functional.cross_entropy(lg.reshape(-1, lg.shape[-1]), batch[:, 1:].reshape(-1))
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
train_s = time.perf_counter() - t0

nll_domain_1, nll_neutral_1 = nll(sidecar, facts), nll(sidecar, neutral)
print(f"обучение: {train_s:.1f} c | ‖γ‖ = {sidecar.gate_norm():.3f}")
print(f"домен   NLL: {nll_domain_0:.4f} → {nll_domain_1:.4f}  ({(nll_domain_1 / nll_domain_0 - 1) * 100:+.1f}%)")
print(f"нейтраль NLL: {nll_neutral_0:.4f} → {nll_neutral_1:.4f}  (Δ = {nll_neutral_1 - nll_neutral_0:.6f}, ожидание 0)")
print("вывод PoC: память помогает ТОЛЬКО на своём домене, вне строк тома — ноль эффекта")

## 6. Роутер темы с гистерезисом

Линейная голова по скользящему окну скрытых состояний. Переключение — только при
отрыве кандидата (margin) и после спада кулдауна: смешанные/шумные входы не вызывают
ping-pong подгрузок тома. Ошибка роутера не ломает систему — промах темы даёт нули
(откат к чистому L1).


In [ ]:
router = TopicRouter(feature_dim=D, domains=["code", "med"], margin=0.5, cooldown=8, seed=17)
a = router.head.weight[0].detach()   # признаки → top-1 «code»
b = router.head.weight[1].detach()   # признаки → top-1 «med»
gen = torch.Generator().manual_seed(3)

switch_log = []
for i in range(60):
    if i < 30:      # шумная фаза: слабые признаки, темы чередуются — гистерезис молчит
        x = (a if i % 2 == 0 else b) * 0.2 + torch.randn(D, generator=gen) * 0.01
    else:           # чистая смена темы: сильные признаки → переключения, кулдаун режет частоту
        x = (a if i % 2 == 0 else b) * 100.0 + torch.randn(D, generator=gen) * 0.01
    d = router.route(x, last_k=1)
    if d.switched:
        switch_log.append(i)
print("переключения на шагах:", switch_log, "| rejects:", router.n_rejects, "| thrash:", router.stats()["thrash"])

# полный конвейер чтения: маршрут → резидентность → gather → Δ
# (холодный старт: реестр в памяти пуст — тома подхватываются с диска после §3)
manager3 = SubtableManager(os.path.join(tmp, "store"), n_slots=1, device=DEVICE)
for name in ("code", "med", "law"):
    manager3.register(L2Subtable.load_pretrained(os.path.join(tmp, "store", f"{name}.npz"), device=DEVICE))
router_r = TopicRouter(feature_dim=D, domains=["code", "med"], margin=0.5, cooldown=1, seed=17)
manager3.router = router_r
ids = torch.randint(0, VOCAB, (1, 10), dtype=torch.long)
r1 = manager3.read(router_r.head.weight[0] * 100.0, ids)  # старт темы → code
r2 = manager3.read(router_r.head.weight[1] * 100.0, ids)  # смена «просится», но кулдаун держит code
r3 = manager3.read(router_r.head.weight[1] * 100.0, ids)  # кулдаун истёк → med; LRU вытеснил code
print("r1:", r1.name, "| Δ:", tuple(r1.values.shape), "| hits:", int(r1.hits.sum()))
print("r2:", r2.name, "| switched:", r2.switched, "(гистерезис)")
print("r3:", r3.name, "| switched:", r3.switched, "| вытеснен (LRU):", r3.evicted)
print(manager3.stats_summary())

## 7. L0-ярус: KV-кэш, выгружаемый в таблицу (KVOff)

Та же механика (LRU-слоты, integrity-манифесты, «диск»), но под спецификацию KV-кэша.
Отличия от L2-таблицы знаний — принципиальны: ключ блока **цепочечный sha256** (семантика
vLLM prefix caching: KV блока зависит от всего префикса), точный тир fp32 хранит блоки
бит-в-бит. Offload-политика удерживает sink+свежее окно, середина уходит в таблицу —
память освобождается. Точный промах при restore не роняет систему: блок честно
пересчитывается поверх resident-префикса (recompute-fallback — «откат к чистому ядру»).
Выгрузка дистиллирует gist-строки в D1-семантике — мост к L2-субтаблице.


In [ ]:
from atomicng.hngram.kvoff import KVBlockSpec, KVBlockStore, KVOffloadSession, KVTinyModel, OffloadPolicy

kv_model = KVTinyModel(vocab_size=VOCAB, d=D, n_heads=2, n_layers=2, seed=17).to(DEVICE)
kv_store = KVBlockStore(os.path.join(tmp, "kvstore"),
                        KVBlockSpec(n_layers=2, n_heads=2, head_dim=16, block_len=4),
                        n_slots=4, device=DEVICE)
pure = KVOffloadSession(kv_model, store=None)      # чистый прогон (без таблицы)
offl = KVOffloadSession(kv_model, kv_store)        # прогон с offload

prompt = [5, 9, 13, 21, 33, 41, 2, 7, 11, 19, 23, 29]
cache_offl = offl.new_cache()
logits_offl, st = offl.prefill(prompt, cache_offl)  # 3 блока: посчитаны и записаны
report = offl.offload(cache_offl, OffloadPolicy(sink_blocks=1, recent_blocks=1))
print("prefill:", st)
print("offload:", report, "| resident:", cache_offl.resident_tokens(), "из", cache_offl.total_tokens(), "токенов")

In [ ]:
def continue_gen(session, cache, logits, n):
    """Догенерация n токенов поверх наполненного кэша."""
    out = [int(logits[0, -1].argmax())]
    for _ in range(n - 1):
        logits = session.forward([out[-1]], cache)
        out.append(int(logits[0, -1].argmax()))
    return out

cache_pure = pure.new_cache()
logits_p, _ = pure.prefill(prompt, cache_pure)
gen_pure = continue_gen(pure, cache_pure, logits_p, 6)
gen_offl = continue_gen(offl, cache_offl, logits_offl, 6)
print("генерация с offload→restore == чистая:", gen_pure == gen_offl)
for li in range(kv_model.n_layers):
    kp, vp = cache_pure.layer_kv(li)
    ko, vo = cache_offl.layer_kv(li)
    assert torch.equal(kp, ko) and torch.equal(vp, vo)
print("KV-кэш после цикла выгрузки/восстановления бит-в-бит равен чистому: True")

# переиспользование: второй prefill того же промпта — все блоки из таблицы, ноль вычислений
cache_reuse = offl.new_cache()
_, st2 = offl.prefill(prompt, cache_reuse)
print("повторный prefill:", st2, "| hit_rate:", kv_store.stats_summary()["hit_rate"])

In [ ]:
# recompute-fallback: «диск потерял блок» → честный пересчёт, генерация не меняется
victim = cache_offl.blocks[1].key
os.remove(kv_store._path(victim))
kv_store._registry.pop(victim)
gen_fb = continue_gen(offl, cache_offl, logits_offl, 6)
print("генерация после потери блока == чистая:", gen_pure == gen_fb)

# gist-ярус: выгрузка дистиллирует «суть» блоков в D1-строки L2-субтаблицы
store_d = KVBlockStore(os.path.join(tmp, "kvstore-d"), KVBlockSpec(), n_slots=4, device=DEVICE,
                       distill_spec=SubtableSpec(name="kv-distill", n_rows=N_ROWS, row_width=D, key_order=2))
sd = KVOffloadSession(kv_model, store_d)
cd = sd.new_cache()
sd.prefill(prompt, cd)
sd.offload(cd)
table = store_d.distill_table()
print("gist-строки:", int(table.occupancy.sum()), "| плотность:", round(table.stats()["fill_ratio"], 3))
print("gist(prompt[-2:]) доступен без KV:", store_d.gist(prompt[-2:]) is not None)

## 8. Что дальше (roadmap этой базы)

- **writer-MLP вместо детерминированного наполнителя** — `writers.mlp.WriterMLP` +
  `WriterTrainer` (BPTT через D1) пишут цели прямо в патч → `L2Subtable.from_patch`;
- **реальный токенизатор и корпус** — заменой `_corpus()`/`ByteTokenizer` на доменные тексты;
- **pre-gated prefetch** — роутер предсказывает смену темы на несколько токенов вперёд,
  менеджер грузит том в фоне (скрытие 1–4 c подгрузки с NVMe);
- **HF-sidecar attach** — тот же инъекционный узел в ранний слой настоящей модели
  (`backends/hf.py`), γ zero-init гарантирует бит-в-бит до обучения адаптера;
- **режим A (kNN-LM)** — интерполяция логитов без адаптера: тот же gather из менеджера.


In [ ]:
print("Итог PoC:")
print(f"  том: {sub.stats()['fill_ratio']:.1%} строк, int8 на диске, integrity ✓")
print(f"  attach: γ=0 → бит-в-бит ✓ | обучение адаптера: домен {nll_domain_0:.3f}→{nll_domain_1:.3f}, нейтраль Δ={nll_neutral_1 - nll_neutral_0:.6f}")
print(f"  LRU: {manager3.stats_summary()['loads']} подгрузок, {manager3.stats_summary()['evictions']} эвикций")
print(f"  роутер: {len(switch_log)} переключений на 60 шагов, thrash = {router.stats()['thrash']}")
print(f"  KVOff: offload {report['freed_tokens']} токенов freed | reuse restored={st2.restored}, computed={st2.computed} | fallback бит-в-бит ✓")